# 📊 NB1 — EDA Transferencias a Terceros (≥ S/2,000)
**Scotiabank Perú · Prevención de Fraude**  
**Período:** Enero – Abril 2026  

---
### Estructura del notebook
1. Carga y validación del dataset consolidado  
2. Perfil del fraude — distribución de indicadores  
3. Análisis de monto (descriptivos + rangos + árbol discriminante)  
4. Segmento de tarjeta  
5. Análisis temporal (hora, día, mes)  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier, plot_tree
import warnings
warnings.filterwarnings('ignore')

# ── Estilo global ──────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

COLORES_IND = {
    'F': '#E63946',   # rojo   - fraude
    'N': '#457B9D',   # azul   - normal
    'G': '#2A9D8F',   # verde  - good
    'D': '#E9C46A',   # amarillo - descarte
    'P': '#A8DADC',   # celeste  - pendiente
}

def fmt_soles(ax, eje='x'):
    f = mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}')
    if eje == 'x': ax.xaxis.set_major_formatter(f)
    else: ax.yaxis.set_major_formatter(f)

print('✅ Librerías listas')

## 1. Carga y validación

In [ ]:
# ── AJUSTAR ESTOS NOMBRES si difieren en tu parquet ───────────
COL_MONTO    = 'MONTO'
COL_IND      = 'INDICADOR'      # F / N / G / D / P
COL_SEGMENTO = 'SEGMENTO'       # segmento de tarjeta
COL_FECHA    = 'FECHA_HORA'     # datetime construido en consolidación
# ──────────────────────────────────────────────────────────────

df = pd.read_parquet('transferencias_consolidado.parquet')

# Normalizar indicador
df[COL_IND] = df[COL_IND].astype(str).str.strip().str.upper()

print(f'Filas: {len(df):,}  |  Columnas: {df.shape[1]}')
print(f'Rango fechas: {df[COL_FECHA].min()} → {df[COL_FECHA].max()}')
print(f'\nIndicadores:\n{df[COL_IND].value_counts()}')

## 2. Perfil del fraude — distribución de indicadores

In [ ]:
vc = df[COL_IND].value_counts().reset_index()
vc.columns = ['Indicador', 'N']
vc['%'] = (vc['N'] / vc['N'].sum() * 100).round(2)
vc['Color'] = vc['Indicador'].map(COLORES_IND).fillna('#999999')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Barras
axes[0].bar(vc['Indicador'], vc['N'], color=vc['Color'], edgecolor='white')
for i, row in vc.iterrows():
    axes[0].text(i, row['N'] * 1.02, f"{row['%']:.1f}%", ha='center', fontsize=9)
axes[0].set_title('Volumen por Indicador', fontweight='bold')
axes[0].set_xlabel('Indicador'); axes[0].set_ylabel('N transacciones')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

# Pie
axes[1].pie(vc['N'], labels=vc['Indicador'], colors=vc['Color'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proporción por Indicador', fontweight='bold')

plt.suptitle('Distribución de Indicadores de Fraude', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_indicadores.png', dpi=150, bbox_inches='tight')
plt.show()
print(vc.to_string(index=False))

## 3. Análisis de Monto

### 3.1 Estadísticas descriptivas extendidas por indicador

In [ ]:
def trimean(s): 
    q1,q2,q3 = s.quantile([.25,.5,.75])
    return (q1 + 2*q2 + q3) / 4

grupos = {'TOTAL': df}
grupos.update({v: df[df[COL_IND]==v] for v in df[COL_IND].unique()})

filas = []
for nombre, sub in grupos.items():
    s = sub[COL_MONTO].dropna()
    if len(s) < 3: continue
    filas.append({
        'Grupo'    : nombre,
        'N'        : f'{len(s):,}',
        'Media'    : f'S/ {s.mean():,.0f}',
        'Mediana'  : f'S/ {s.median():,.0f}',
        'Trimedia' : f'S/ {trimean(s):,.0f}',
        'Desv.Std' : f'S/ {s.std():,.0f}',
        'CV%'      : f'{s.std()/s.mean()*100:.1f}%',
        'P25'      : f'S/ {s.quantile(.25):,.0f}',
        'P75'      : f'S/ {s.quantile(.75):,.0f}',
        'P90'      : f'S/ {s.quantile(.90):,.0f}',
        'P99'      : f'S/ {s.quantile(.99):,.0f}',
        'Máx'      : f'S/ {s.max():,.0f}',
        'Asimetría': f'{s.skew():.2f}',
    })

tabla = pd.DataFrame(filas).set_index('Grupo')
display(tabla)

### 3.2 Rangos de monto — concentración por indicador

In [ ]:
# Rangos sugeridos (ajusta según lo que veas en los descriptivos)
BINS   = [2000, 5000, 10000, 20000, 50000, np.inf]
LABELS = ['2k–5k', '5k–10k', '10k–20k', '20k–50k', '50k+']

df['RANGO_MONTO'] = pd.cut(df[COL_MONTO], bins=BINS, labels=LABELS, right=False)

# Tabla: N y % por rango × indicador
pivot = (df.groupby(['RANGO_MONTO', COL_IND])
           .size().unstack(fill_value=0))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras apiladas (volumen absoluto)
orden_ind = [i for i in ['F','G','N','D','P'] if i in pivot.columns]
colores_orden = [COLORES_IND.get(i,'#999') for i in orden_ind]
pivot[orden_ind].plot(kind='bar', stacked=True, ax=axes[0],
                      color=colores_orden, edgecolor='white')
axes[0].set_title('Volumen por Rango de Monto e Indicador', fontweight='bold')
axes[0].set_xlabel('Rango (S/)'); axes[0].set_ylabel('N transacciones')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

# Heatmap % fraude por rango
if 'F' in pivot_pct.columns:
    fraude_por_rango = pivot_pct[['F']].rename(columns={'F': '% Fraude (F)'})
    sns.heatmap(fraude_por_rango, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.5, ax=axes[1], cbar_kws={'label': '%'})
    axes[1].set_title('% Fraude por Rango de Monto', fontweight='bold')
    axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('fig2_rangos_monto.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n% por indicador dentro de cada rango:')
display(pivot_pct.round(1))

### 3.3 Distribución visual del monto (KDE + Boxplot + Violin)

In [ ]:
orden_plot = [i for i in ['F','G','N','D','P'] if i in df[COL_IND].unique()]
pal = {i: COLORES_IND.get(i,'#999') for i in orden_plot}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# ── KDE superpuesto ─────────────────────────────────────────────
for ind in orden_plot:
    sub = df.loc[df[COL_IND]==ind, COL_MONTO]
    if len(sub) < 5: continue
    sns.kdeplot(sub, ax=axes[0,0], label=ind,
                color=pal[ind], fill=True, alpha=0.2, linewidth=1.8)
axes[0,0].set_title('Densidad del Monto por Indicador', fontweight='bold')
axes[0,0].set_xlabel('Monto (S/)'); fmt_soles(axes[0,0], 'x')
axes[0,0].legend(title='Indicador')

# ── Boxplot ─────────────────────────────────────────────────────
bp = axes[0,1].boxplot(
    [df.loc[df[COL_IND]==i, COL_MONTO].values for i in orden_plot],
    labels=orden_plot, patch_artist=True, notch=False,
    flierprops=dict(marker='o', markersize=2, alpha=0.3)
)
for patch, ind in zip(bp['boxes'], orden_plot):
    patch.set_facecolor(pal[ind]); patch.set_alpha(0.7)
axes[0,1].set_title('Boxplot del Monto por Indicador', fontweight='bold')
axes[0,1].set_ylabel('Monto (S/)'); fmt_soles(axes[0,1], 'y')

# ── Violin ──────────────────────────────────────────────────────
sns.violinplot(data=df[df[COL_IND].isin(orden_plot)],
               x=COL_IND, y=COL_MONTO, order=orden_plot,
               palette=pal, inner='quartile', cut=0, ax=axes[1,0])
axes[1,0].set_title('Violin Plot del Monto por Indicador', fontweight='bold')
axes[1,0].set_ylabel('Monto (S/)'); fmt_soles(axes[1,0], 'y')

# ── ECDF Fraude vs Normal ───────────────────────────────────────
for ind in ['F', 'N']:
    if ind not in df[COL_IND].values: continue
    s = df.loc[df[COL_IND]==ind, COL_MONTO].sort_values()
    ecdf = np.arange(1, len(s)+1) / len(s)
    axes[1,1].plot(s.values, ecdf, label=ind, color=pal.get(ind,'#999'), lw=2)
axes[1,1].set_title('ECDF: Fraude vs Normal', fontweight='bold')
axes[1,1].set_xlabel('Monto (S/)'); axes[1,1].set_ylabel('Proporción acumulada')
fmt_soles(axes[1,1], 'x'); axes[1,1].legend(title='Indicador'); axes[1,1].grid(alpha=0.3)

plt.suptitle('Análisis Visual del Monto', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_distribucion_monto.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Árbol de decisión — ¿en qué umbrales de monto discrimina el fraude?

In [ ]:
# Solo F vs N+G (etiquetas más limpias)
df_tree = df[df[COL_IND].isin(['F','N','G'])].copy()
df_tree['TARGET'] = (df_tree[COL_IND] == 'F').astype(int)
df_tree = df_tree.dropna(subset=[COL_MONTO])

X = df_tree[[COL_MONTO]]
y = df_tree['TARGET']

arbol = DecisionTreeClassifier(
    max_depth=4,              # 4 niveles = splits legibles
    min_samples_leaf=50,      # evita hojas con muy pocas obs
    class_weight='balanced',  # compensa desbalance F vs N
    random_state=42
)
arbol.fit(X, y)

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(
    arbol, feature_names=['MONTO'], class_names=['No Fraude','Fraude'],
    filled=True, rounded=True, fontsize=9,
    impurity=False, proportion=True, ax=ax
)
ax.set_title('Árbol de Decisión — Umbrales de Monto que discriminan Fraude',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_arbol_monto.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar los umbrales que encontró el árbol
from sklearn.tree import _tree
tree_ = arbol.tree_
umbrales = sorted(set(
    round(tree_.threshold[i], 2)
    for i in range(tree_.node_count)
    if tree_.threshold[i] != _tree.TREE_UNDEFINED
))
print('\n🌳 Umbrales de monto encontrados por el árbol:')
for u in umbrales:
    print(f'   S/ {u:>10,.2f}')
print('\n→ Estos son los cortes naturales para definir rangos en las reglas.')

## 4. Segmento de Tarjeta

In [ ]:
if COL_SEGMENTO not in df.columns:
    print(f'⚠️  Columna {COL_SEGMENTO} no encontrada. Ajusta COL_SEGMENTO arriba.')
else:
    df[COL_SEGMENTO] = df[COL_SEGMENTO].astype(str).str.strip().str.upper()

    # Fraud rate por segmento (solo F vs F+G, etiquetas limpias)
    seg = (df[df[COL_IND].isin(['F','G','N'])]
           .groupby(COL_SEGMENTO)
           .agg(
               N_total  = (COL_IND, 'count'),
               N_fraude = (COL_IND, lambda x: (x=='F').sum()),
               Monto_medio_fraude = (COL_MONTO, lambda x:
                   df.loc[(df[COL_IND]=='F') & (df[COL_SEGMENTO]==x.name), COL_MONTO].mean()
               )
           )
           .assign(Fraud_Rate = lambda x: x['N_fraude'] / x['N_total'] * 100)
           .sort_values('Fraud_Rate', ascending=False)
           .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Fraud rate por segmento
    colores_seg = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(seg)))
    axes[0].barh(seg[COL_SEGMENTO], seg['Fraud_Rate'], color=colores_seg, edgecolor='white')
    for i, row in seg.iterrows():
        axes[0].text(row['Fraud_Rate'] + 0.05, i,
                     f"{row['Fraud_Rate']:.2f}% (N={row['N_fraude']:,})",
                     va='center', fontsize=8)
    axes[0].set_title('Fraud Rate por Segmento', fontweight='bold')
    axes[0].set_xlabel('Fraud Rate (%)')
    axes[0].invert_yaxis()

    # Volumen total por segmento (barras apiladas F vs resto)
    pivot_seg = (df[df[COL_IND].isin(['F','G','N'])]
                 .groupby([COL_SEGMENTO, COL_IND]).size().unstack(fill_value=0))
    orden_ind2 = [i for i in ['F','G','N'] if i in pivot_seg.columns]
    pivot_seg[orden_ind2].plot(
        kind='barh', stacked=True, ax=axes[1],
        color=[COLORES_IND.get(i,'#999') for i in orden_ind2],
        edgecolor='white'
    )
    axes[1].set_title('Volumen por Segmento e Indicador', fontweight='bold')
    axes[1].set_xlabel('N transacciones')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

    plt.suptitle('Análisis por Segmento de Tarjeta', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig5_segmento.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(seg.round(2))

## 5. Análisis Temporal

### 5.1 Fraud rate por hora del día

In [ ]:
df_clean = df[df[COL_IND].isin(['F','G','N'])].copy()

hora_stats = (df_clean
    .groupby('HORA')
    .agg(
        N_total  = (COL_IND, 'count'),
        N_fraude = (COL_IND, lambda x: (x=='F').sum())
    )
    .assign(Fraud_Rate = lambda x: x['N_fraude'] / x['N_total'] * 100)
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Volumen por hora
colores_hora = ['#E63946' if fr > hora_stats['Fraud_Rate'].mean() + hora_stats['Fraud_Rate'].std()
                else '#457B9D' for fr in hora_stats['Fraud_Rate']]
axes[0].bar(hora_stats['HORA'], hora_stats['N_total'], color='#ADB5BD', label='Total')
axes[0].bar(hora_stats['HORA'], hora_stats['N_fraude'], color='#E63946', label='Fraude')
axes[0].set_title('Volumen de transacciones y fraudes por hora', fontweight='bold')
axes[0].set_ylabel('N transacciones')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

# Fraud rate por hora
axes[1].bar(hora_stats['HORA'], hora_stats['Fraud_Rate'], color=colores_hora, edgecolor='white')
media_fr = hora_stats['Fraud_Rate'].mean()
axes[1].axhline(media_fr, color='#E63946', linestyle='--', lw=1.5,
                label=f'Media: {media_fr:.2f}%')
axes[1].fill_between(range(0,6), 0, hora_stats.loc[hora_stats['HORA']<6,'Fraud_Rate'].max()*1.2,
                     alpha=0.08, color='#E63946', label='Madrugada (0-5h)')
axes[1].set_title('Fraud Rate por Hora del Día', fontweight='bold')
axes[1].set_xlabel('Hora'); axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_xticks(range(0, 24))
axes[1].legend()

plt.tight_layout()
plt.savefig('fig6_hora.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 5 horas más peligrosas
print('\n🕐 Top 5 horas con mayor Fraud Rate:')
print(hora_stats.nlargest(5, 'Fraud_Rate')[['HORA','N_total','N_fraude','Fraud_Rate']].to_string(index=False))

### 5.2 Fraud rate por día de semana y mes

In [ ]:
ORDEN_DIAS = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
NOMBRES_DIAS = {'Monday':'Lunes','Tuesday':'Martes','Wednesday':'Miércoles',
                'Thursday':'Jueves','Friday':'Viernes','Saturday':'Sábado','Sunday':'Domingo'}

dia_stats = (df_clean
    .groupby('DIA_SEMANA')
    .agg(N_total=(COL_IND,'count'), N_fraude=(COL_IND, lambda x:(x=='F').sum()))
    .assign(Fraud_Rate=lambda x: x['N_fraude']/x['N_total']*100)
    .reindex([d for d in ORDEN_DIAS if d in df_clean['DIA_SEMANA'].values])
    .reset_index()
)
dia_stats['DIA_ES'] = dia_stats['DIA_SEMANA'].map(NOMBRES_DIAS)

mes_stats = (df_clean
    .groupby('MES_NUM')
    .agg(N_total=(COL_IND,'count'), N_fraude=(COL_IND, lambda x:(x=='F').sum()))
    .assign(Fraud_Rate=lambda x: x['N_fraude']/x['N_total']*100)
    .reset_index()
)
MESES = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril'}
mes_stats['MES_NOMBRE'] = mes_stats['MES_NUM'].map(MESES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_dia = ['#E63946' if d in ['Saturday','Sunday'] else '#457B9D'
               for d in dia_stats['DIA_SEMANA']]
axes[0].bar(dia_stats['DIA_ES'], dia_stats['Fraud_Rate'], color=colores_dia, edgecolor='white')
axes[0].set_title('Fraud Rate por Día de Semana', fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)'); axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(mes_stats['MES_NOMBRE'], mes_stats['Fraud_Rate'],
            color='#457B9D', edgecolor='white')
axes[1].plot(mes_stats['MES_NOMBRE'], mes_stats['Fraud_Rate'],
             'o-', color='#E63946', lw=2, ms=8)
for _, row in mes_stats.iterrows():
    axes[1].text(row.name, row['Fraud_Rate']+0.02,
                 f"{row['Fraud_Rate']:.2f}%", ha='center', fontsize=9)
axes[1].set_title('Fraud Rate por Mes (Ene–Abr)', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')

plt.tight_layout()
plt.savefig('fig7_dia_mes.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Heatmap hora × día de semana (el más potente para reglas)

In [ ]:
pivot_heatmap = (df_clean
    .assign(FRAUDE_BIN = lambda x: (x[COL_IND]=='F').astype(int))
    .groupby(['DIA_SEMANA','HORA'])['FRAUDE_BIN']
    .mean() * 100
).unstack(level='HORA').reindex([d for d in ORDEN_DIAS if d in df_clean['DIA_SEMANA'].values])

pivot_heatmap.index = [NOMBRES_DIAS.get(d,d) for d in pivot_heatmap.index]

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(pivot_heatmap, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Fraud Rate (%)'})
ax.set_title('Fraud Rate (%) — Hora × Día de Semana', fontsize=13, fontweight='bold')
ax.set_xlabel('Hora del día'); ax.set_ylabel('')
plt.tight_layout()
plt.savefig('fig8_heatmap_hora_dia.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n→ Las celdas rojas son los candidatos directos para reglas temporales.')

## 6. Resumen ejecutivo del NB1
*(Completar después de correr el notebook)*

In [ ]:
fr_total = (df_clean[COL_IND]=='F').sum() / len(df_clean) * 100
monto_med_fraude = df_clean.loc[df_clean[COL_IND]=='F', COL_MONTO].median()
monto_med_normal = df_clean.loc[df_clean[COL_IND]=='N', COL_MONTO].median()
hora_top = hora_stats.loc[hora_stats['Fraud_Rate'].idxmax(), 'HORA']

print('=' * 55)
print('RESUMEN EJECUTIVO — NB1')
print('=' * 55)
print(f'Fraud Rate general        : {fr_total:.2f}%')
print(f'Mediana monto FRAUDE      : S/ {monto_med_fraude:,.0f}')
print(f'Mediana monto NORMAL      : S/ {monto_med_normal:,.0f}')
print(f'Hora de mayor fraude      : {hora_top}:00 h')
if COL_SEGMENTO in df.columns:
    seg_top = seg.iloc[0]
    print(f'Segmento más riesgoso     : {seg_top[COL_SEGMENTO]} ({seg_top["Fraud_Rate"]:.2f}%)')
print('=' * 55)
print('Figuras guardadas: fig1 a fig8 en el directorio de trabajo')